<div dir="rtl" align="right">

# ورقة عمل الأسبوع ١: NumPy وتنظيف بيانات عربية حقيقية

**الهدف:** تكتب ١٩ دالة: ستاً في NumPy بلا حلقات، والباقي لتنظيف طلبات متجر كتب عربي فيها كل ما ستقابله في بيانات حقيقية بالمنطقة: أرقام هندية، وتواريخ بخمس صيغ منها الهجري، ومدن مكتوبة بأكثر من عشرين شكلاً، وتكرارات، وقيم غير صالحة.

**الوقت المتوقع:** ٥–٧ ساعات موزعة على جلسات الأسبوع.

**طريقة العمل:**
1. شغّل خلية الإعداد مرة واحدة.
2. في كل تمرين دالة فارغة. اكتب حلك مكان `raise NotImplementedError` ثم شغّل الخلية؛ الفحص في آخرها يطبع ✔ أو ✗ مع تلميح.
3. `W.progress()` في آخر الورقة يعرض ما أنجزته.

**القواعد:** المحاولة الأولى من ذاكرتك ومن وثائق NumPy وpandas فقط، بلا مساعد ذكي. بعد ✔ اطلب مراجعة لأسلوب الكود. الحلول في `week01_solutions.ipynb`، فلا تفتحه قبل أن تحاول.

تعمل الورقة على Colab وKaggle وجهازك (pandas 2.2 أو أحدث).

</div>

<div dir="rtl" align="right">

## الإعداد

خلية اختيارية لتثبيت أداتين لرسم العربية في matplotlib، ومكتبة التحويل من الهجري للتحدي الإضافي.

</div>

In [ ]:
# Optional: Arabic text in plots + Hijri dates (bonus exercise)
%pip install -q arabic-reshaper python-bidi hijridate

<div dir="rtl" align="right">

خلية الإعداد: تولّد البيانات وأدوات الفحص في الكائن `W`. الكود مضغوط حتى لا ترى الإجابات مصادفةً، ونسخته المقروءة كاملة في ملف الحلول.

</div>

In [ ]:
#@title خلية الإعداد: شغّلها مرة واحدة
import base64, types, zlib
import numpy as np
import pandas as pd

_SRC = (
    "eNrFfeuSG8ex5v95ijrt0HG3DUAA5kZiPY4dUZRFH4nSUpR1gRFQD9CYaQ1u7O4ZcjQ7Ct1v3rDkExvriN2N8EVHIkVZkqm7vD82"
    "YvclgOA/vcC+wn6ZVdVd1d0Ah7rEoRSYRldVVlZWVlZmVlbCcZwngmBfNER8MJ1OokSMJv2DYdASoyCOj8R25O+EPbEzmezHySQK"
    "RN9P/IqIgkEQBeNeIPxxfDWI4goe+iIOhoNqby/o7ce1lZXH40CEiRhEk5FI9gJxdRLtx3tBkAg/Fs888UxrReBf5F8VW+KJ2nDi"
    "97v44nrC+PcTbilxoa7F0eRAXA2HQ9EbBv6YITxR4y67z8U9YOjKP14GIToYMxSJmBhMIjEZByK4FkS9MA4UjGk02Y3Qj92/hhEH"
    "wH/PT7j/Pf8wEINwHGI0/ZWV84dBdJTsheNdEcaiHyRBNKLCBIRzB+G1gAgT9L2KiCcmIn5ESPi9pLbyBJEG40r2xBSEBH2atebP"
    "mabq+2rtWkWMD0bTI9GoNTdkWbN2rbbiOM5KOOK5i4IV/SirouF4ql8pSPh/Cqx/Iqo/3D9AOzcZx4k/TuIfGPL2pe79F3514fJj"
    "4BJn/uf5X+Z/nb87/7f5e/P359fnN+YfOCsPbBtVbv/99q3bn97+7Pbnt7+4/eXtr25/ffsbZ0VW6F7evu+h86gVJ1Ft5O8HSQT+"
    "dZkDjn/2s2O/RSVu6DGThGBqEY5FAFoGkZ8EboqLd1LhRgKNBvlGA7tRil3ayJl/6LSEU3MqePwbPTonK95K9/Ij3e1LBeyceqO5"
    "ura+sXnmLBpkKKysnLtw+anuue2Lj1xEo7Yzuz5/ZXZr/tbs+uxLAj372+yT2Qf8RCWfzF/F31e501fnL88+EPSaH2+hKK2I50/m"
    "b81fyyq8Nn9jdktVeHd2Y/767LrTkb0/Sj3Xa6vNiqjXmvzZWONP+VyXn6r2b7YvXdi+yPN0zKSwcW6VDcL4LuiFWFRj/gL9l741"
    "alwKj/z+Hj9deGr7/gfpKeJ3DPHS4w86nYrCR9KMMcnIh6f56/IJPai3vw76fZ+h/vr8/fdvP5i+U68ymCb10zEa02E9qzIaiXxz"
    "vz8a+SN+evjhFGj5FBL0hZNL3+avWwVyVLJFVoWeHvb39+XoHg56Pd8azGIO0aNbzkNGhQwfrqCobEGQ2PTDsUaHHjN8NENy1/oL"
    "AzGe6f0L+tv2zh63P5FM+evHL/4Lr5+L925z5T/N/kR/q/TBL/4BVrolgNFHmPtPwPoQcS5tid2wXxE9fzwZhz1/KJIwGQYVcTDG"
    "rjeNQuyOEASPbV/yIBAhSgZhLwknY1SUW2kP+9lwsnsQrNz3yCP/QouizWNynfvqDZOHJct/wZ8fEVFQuLZeq2NHobpNrvsFFc0+"
    "knVvzD5GQzRHUeNMnaqmoFfVRL/EbEcTBOK8JmYfM+O9wdA31mvrCvoaV38ZM/IWKnwgME3EnnJiVm3Q65LYqPuiMBroR0KmWU/x"
    "3shWLw/rJcUKwP0mGhHzr56x4G9yi5vzF2ef8ZQo1MG9H9J3lDbXMvhnUvjvEh0ZYHOTBpYCPJtWucFVXqPh0deb9EKRerOpQTbq"
    "XP89JvX12V/E7BsgDFSFWrayQWPdIkujkSECBrqhOf5FjPsTGmQ6lQ2eStT7jHHhJt+g1g2a8KYNlKdx9hV6l3QDTixEmMzNMynI"
    "tXRFvUzF81fE/MX5W0KNkplk9jE1adjweS6B482MpzRHrZ5NidyQk/gpyj7R0/cNfcHrs+sWwE2DCv+g8TGs5kYG60y2+MGdNJbr"
    "s5vU36o5aQ05abeYSK8oCYM1SuS8zvRvbmiQTTlhH9GAmVPAZbOvdTWe1xVezr+Kgt1JFPpjUf2leDB8NgqF+/hoJPxh9T8dRD6g"
    "9Saj6UECXY61NFLjntmjetBKg2egW/X2/d2gtvLghV9futB9ePvRbI9r1pvr1XqziqWErb6xtrZRrZ+pysWiy5obZllzMytbQ3Fa"
    "1qgTlIoJF+WbVvlZsy041iiT/JWWrVtlmzZc+j8rb1TrBk7r1ebZrKxZrRtwN6x2TcbHhLuZ0QGPDQvfTcLQKGsYfZ4F6KxstdrI"
    "0eFsRgdZntEX2Kd02CTsG7rPDapZN/o8W6WFlcLdYDqdTcuJDs2s7Vq1uWaVbVpl61bZGRuuQV8ul9wvyzazsZwhOjSbZlmznpU1"
    "iSoW3M2sXy6X0ofLaJY3sjLGgLbBlQcuXNx+qHvukYcef/gi70POJOoHEbY3aiufic/pWy9Mjuiv2v/SR978uuNJNFL4WP8c3g+p"
    "8pUDGAkKBvRjWE38FBxCYw7k3vrDGiaXUlsVdtAwGAXjxKd9OBYurWJpwIr94Og/iIMY63vnyDDSvB8YHSjv2+dA4SiokUAJh4Hr"
    "tGfvVGf/ff676vxP87/f/m/V2x91HFgDFy7+5sJjF6S9YlX/9oXfVb994e1vX7iJPx9/++L71W9f/OD//a//za22Hzr/QMGEOMam"
    "TPOutKH3jOe/Zs/zW+r5BLbFSj8YiC7sfG1WP+lJe/1JQB9Pa37sR5F/5D5ZEf3kaBpsDWDAJx5XGR3A1O2j3pO1EYx0178Wxlsk"
    "kZ+sxUlff+Wqk4NEwnsuiCZxdxjuU09cNH6OxtEXvxR1XbXdgg38XAfv3Sf1cxXdtfHgiXtRm56kWyFIDqIxNTLHMvXD6Cpsfne7"
    "Iu5TA6JHe0x4I0dTMd/ep99yKxqeuw37TzS9WnwwchseYXQRnoWO+Llw77OL6H1FtAjdpviZ2Bb/UdxXu2xiip5GoMwI9ftkNNWt"
    "SYgngwTF7tOwSYl81YbC/mkb9actJANZGFybuk+j56epB0l++qiA6YNpPxzFW5ejg8DzTHQCkDNg7BfWN/HD8Lp7k8Q9YidFtzf0"
    "4ziIFY5HNo5HZVPvuke1OHwusNunNdvUHry8G6h6mJsjYoTGktkeTQ4hXrrw1kTYm104UPYVRj3Zc+9gRCPEUziOgyhxDSyvZUxQ"
    "l/Nh0cfttfd5OnvtVnWfuW/f7HufOD/uxkkwpTVyTq8ePBYWUAmznbPmcejvBMMY7WwmZrggyy7cTWAyuWqCq9TBOciL6ZEr35Ff"
    "4lmyRSQBz9XiPX8atOsdhRSvWTTS3WyJZ9P34UCMav4YoFqWVKd+2s/SDDzZHnXkOq9bFJLQKlzTpEwy6fpxLwy7/XA3TGJXc4lq"
    "Nu3XHguiMECBFi3OZOfZoJc4Hvh36g790U7fF4ctcVhj8TYkL4vp4PEI6zAO2SHVC9zDCslDTwCdgIg89scW87Irscs7VIoNFYKp"
    "3UNj3IA6niQlkG3SZAt6rDyU9O8QpFqIMORiFE5dD15QbEZq2lSXsNXggCCVlabwsLQvsGf6mmY7mexTZRdqcm32WWZLzl+hZ/CY"
    "k8NZYhcF06GPYaE52b7eSnmxw24ToziJjkrRYhYGCdOy4FovmCbiN/7wIDgfRZPoTpQ7LVNgqrASYipxHe52Y83xbNEfxQFrMRnL"
    "JXpJFXmSZkTPiiGuMkTwdNG/XAGZ+8G1raTGf1PkqJ8kHAUba+1x3FGkCuH/3RIJgx4cDIcjP+ntuZHz2/7x2kkVn031CfKO/a0H"
    "fPBr2rKbnK6t4M9WOZz+6GgRlObJvennWrElyWBg0ZE0AL30CF35vkJ8B3Bbzj1PVe8ZVe/pO1bDbrKoKUoksYe0/iBrG6QqLIGG"
    "QZTCovdGw/6994zuvecpxyvuEF1ppcHf+qtL539FltpeS+zyytmtiD1aOqkdVwuTYIRTgROTmdjyI67Z1XZjylM/EZctrZKUypg1"
    "ysTfGaJkZ3IY5I8CxjgfgGzB+oCAyIxJCe+pyQFOLK6O0WZ4QJqriPcmB8M+QYZbKQrgUepNxtjlcPLQEtNwKlhA4ZAktVD/Pdh9"
    "dDqGnb8+f6HAbsqVfkhTkdAGo2YhkxfMV4oPLqNrDHg0de2ZbR+2W3A+d7wlSgJZLP4QGkXX57OuH3cHyLR66FY7LvkXD20xyxbC"
    "okJDBjPRijLa2GDIFMiKFU6Q3bVnJ9AXDmvxdBgmrq3WnErQ0gpi1+lD8Fs+zp6OkxWasS77QiuiewhhS1Nnef3tOeT6PL9cOyOg"
    "CbqdotstDq1Ii3RIeh8l/pBIWfu99th2yZhNJ3ysl0cJS1iaek4NMRGu7QbY8HT3Fa1tfCeFRNvgd60hpaQ0cCzju1TrwNEvoXuX"
    "WPaD/gG22/5AoSdFRn9Qi3HaiEnFHh+7tivhYEqSot/1EzjgxT6EyJaDY7LdgJoUJHWtH02mXXSCnYH3bRMW2SJbDuyExCmhXThO"
    "iDrDCcR5+O+q0i1WkgZo/v01JFPdouG4w4n4BQDTx17Ix8QDUumJIgPPms0fQMNK31wYFzQuORPa35Pn4uJkNSpivV4CQXqJTtfe"
    "ah6yx1Yp9kU2NSykUUD2PLHcbjQ5mO4cual/y2sr31VHzjTpFy6Yto9N31BKdCUJJmszQKzC2HdRf9kmZCz42EUQREXu3NLOVtxL"
    "7IIiE+3+wFheHS2+ctLDrpQ1NNdiUZ3K1/AyJHKLXxbQ+swM1EzVznpnB6KCAz7lAWbMrQHwX021RbpWAWhOZ7kfJ2sPRD6GcZx2"
    "kNGglaNbJV+HgbYkKkYhez5bpVsIAeTijmc00CwkO9TfOvkahue0tWADSttzXbsXyWmtoilLjRQbmvVTB2yrdJlSq7SK1VB5a1sl"
    "a5MaqWLd5MSck+JyxHtj8aTu39zygauMv2YIlRh5uU3D3H2yWSZtIQ6SLuutLm0scnG1Ld93x3YfjZO94RHPcFdhmImRvhQiBAmc"
    "ik0V0LfaZc5yW/3s16bh4STpsjXgSnW6b7NzrZ/QWsSeHE76rvNwJmZpZ6KjqOHBaBxvKV980dueblBMha2UvHAe7u4ODsa9LQd+"
    "L3yldSZptZX3OCbYejnsC0OGHrO1rsa9K8dty0mDhT1jNtl/6KXNdq058HJ6QoZkASZ0BdgrwRgn/7tbbTYTKoKmzybtbm0v8Pvu"
    "GJDTXpbit/LDnzaQ6BG7wZhifybRDwyeJ4eC4eSAXBVwEI13DYcGv+RvpF/RpjHehbmWBLu0ucC+3khlsKwAJ2qL4+SgKyBUDxtb"
    "LwoRvBanjNXbI30e50Idy8vUY0s5sTUUrlvzoVGO+25vz7MKyZ01+4tDugka/4Iiut5yWEchJCUC0EN/AYfW6nqrwNgWbGrR25uQ"
    "MEHQy+/5BONt/nyHP/+VP/+Q7g6aPo6yKBiY8pMPDWI0mRg4I7oaBEOypMN+wLb6cALvaZxQHGM/hUhfYmnosu0BgyXr7lkh9S52"
    "Xu+S8709DMbuVRkqdpWox+1NFCkakl/Cu2qqqarhL8WqTRddl5q1W006fHAoEMrBw9V2s9WxB6/tKW5WMvpVHr0/BKfBLAs5oM8E"
    "kGRWF86U+LzeMSwxnC2pd4RzYVLXpfppAlE+yfkt/tsoQWmNUZIxoDF8I0GcG5KgsSY59yTtWB4RQzhFkOsMMhwfQvEnp0wU7u4l"
    "1WRSxbgRAutH+/agCQ6O3BxT2qQK3GCUdHFosQMT6hovRrIgjoZacSPuGDjH11q15uDEWUYWl6vtnjjpAmU4hLADl2027wQyNg0P"
    "jhw0pkGGFn5ob46xQpjOBWmn8SElXYpM3dokAY/QnLpCGSGll9QxByujOECCIRD0yf0zmHg18Yx+8wyFvHJgrI9tKYrwUkbmUjQs"
    "PFGgOkJtEVvbP8C81ShWlTEa78pTEEmGGtDyD4ZJNyI9P9C6csgrixcPnYm0B84jl+6vHjfq9A9TEoKc7CjKzjbGntoSYM1FCTBn"
    "CQgQmYfKVQEDOJJvONJEzxXySflqQ1EPLDdW69glcFUJO92TUeARozUsRZjr4G1e+BI09JkZTtRvPxgmPnxn92vXGVhKCe5UwsFT"
    "SpMKJyf98aHUyBfsQQUMemZVmXZLOiHbQj/TLQRnrnNsJgdiZh+KTGjRhZPSojNty0HfTT2gXpmbL0WVIIT9axLA1T0ctbuM/ZZG"
    "x8NBU0aatqrfscen8VCok7BTFZVohKoTMmGNRln8qzliGZcqWymd+1rJLkhdcNwdsYCsfSU5ylG9wVYl/m/y/6sIw4KRmRJYLS9W"
    "fIstSxqoGanzlNTP8ExsVni3k5+dlDGM4z/ZQdsQG2MpN5rrHUl3bcwPoiBQiNg165I7ZVSiuaaYBHCktrFzpGtJk00xCbdqE2zq"
    "Tp81Keswtdk0QxtMMuqwCLZIv1GniDoo9vSwYB1QuxV9O6AbTa7GLIP4idWQilZFCsvfcBCHhBtkCEaXeUps6nhm7a68ikD7B5Cm"
    "5rx7yJ1MUlELalVIW4JR3KjLYpRqD6BRuJ4WZkavjNRw5coIO9a5H1YRO7i3eI079q6vkc1g8f5mtCFBUd4Gu0y/fYYc4yf34mm9"
    "tSkf6q21zomzEKASOQvwKO5GCwGlEmsxeifi2GIarJTmqteqN/snreOCPluXJSbqcWAD3wPgVKC1+1JLooOHUhT2iqNZtm/vmdNW"
    "qFTPKWzSnKRtlZ/A14oL8pKNApO91Ie8ZHCnhqg98W0pTTEhHeog/ZZC3aHAZmVddAFMSgktFqhdph+xaSXptsA2sbgaUgTt7QGw"
    "gFEw0uNuWAY2CxUpu74YjIq4yQVwU1wx4kSdTjvPRKteZwmFlR5W2ETWbfOG6C6FpUHLAmamwjhVCqMTwJm4xJIvbUPH+Cyj1Cn/"
    "4uYOotBZqmFxTaUiuqR2WklkUQN3iRoNh1FDx6A206+zfJms2SS/Ijn6Sjk7tx2O7K1ycGNDhvmqqP8PEZtO0RKdU62dK0q1w95v"
    "CeAUBZox6vyKxwcD+FsUDhUi2ZWTGqExcAQeRQmLrXmdHHg5tCvWaQRikWKg7KpNP+zkjyIkXSLVOOXzYn9Nr1NCgNOIqFwf5bTf"
    "UPcIXuUw8k9PSe/oUBE8G1+xa03z6FARHQ8LqB4dEtmXr2cJV44lOkxfQ4FRg1OqDCGT2VCX2YqUbNyq1zNhpHUS7Yg4Nl3LSo+o"
    "5H3J6qXyIWcS23ASk9BdsoQN/28rk7qVzAGcCpmK5eSVC2kJ4NSvq6bd9Pu3iEon5vHUdOnY8+O2DufpCqRFAUD7bhRY7i1vcx3M"
    "ZTic9GB9LIOpiScP4rTw9vIklJgalKIXS8DKUwuCm+k98hXAdBWB8/TBa+9EKb4yEiR1ueGwxbBj1EIcK7OClhMtYijVY48u5zIH"
    "F8IpSE1WcNj9wXdPRXq8KmhejYlG/3qW+7i55Gq2J/KkNjamoojWlPyv/XAwaPTdLG6TUFPdezbizVMgnvYl/VNw9Q/BZgJBLzFF"
    "xEwG7KKLceYjmAEreJUEY3lbwxeEDUWBJ8MjcRXeTSoyFHGMi5TY/DDN8txJm7QrLDV/gXYozYmcHSKxzwWvcC/WoRucDaoGm0Yu"
    "4n6imMlmyTsYm2tnPC+Hb+6AjwQdd5urlp2tqEoF8Bsck5UHr897FrVqkKfZ88o5ChA84/q3dWanZwBU4zZmRToAxSQjjt8dRH5v"
    "q1GDrJdUR6wthrvF7qMFhz16aU3zXWqx5ukK9oEMqhfeldm7WewTV7com3tziqP0FI5Baev7XcEwTtj4RXbCdkfMDKrh8y4P2KQn"
    "D25DChOSMhDVY8g+cj8SPBLSjEM39V+ZB9a2mHW0INAQaGGlwoEFtxQyZrkSOwVYmRzmpf8c4jWMeeazWkmdVGJD9J3kyEFDdfWR"
    "XDoIU8x3CqeMhldWO1ilb5VDq7bPPXheBlUpDzM5a5WnVmmD5BuWMYMJh1BxGzOGir63qQpNeMHja/mFzcqqxzQZg9UpvLeX02wM"
    "MlhBBTXaWRmE65OdF+8JilUQAeVHwG3X4dDLHMCq63Rk5CXUoQ0GDtrPXIqIDLJspYkZohLXswqVlP3VBOWkmIyxCfjYHSLGWyZy"
    "WIZZw8Dshz4vfHSIGdwLhjjs5c1OJdxIIPloYhE1OUWNYbjzIxwl+pFL/WREfYzuBcjjuIDn10IHIasZOqIP9ogpv4dy+w+PZLzq"
    "RZAstqNQpVpWJY7waZxTZMuYjKvQ9nDLcoAasdhByCui1nkTJ0nPeKVzYoVOqWwWEmhXA80OJindCIGu+UMKF0n2RroJYvMQ8Ypz"
    "Ov8oHzplFLk5yDX1QIfvkl6elogcqnWe/0AJaeWBpg1+BL45R4HDmI8e8o5gef3Q19QunX/s8YdkooaTlS6OXc5f4nuB2w0h74Px"
    "Rfqm0HdR+OuqUNeU+NuaUJeC+Nu6sG/j8MsNYVyTsXwMuOYrclHK1OK+pjAiXPjNqjDCjeSbn8vgIrqjPz6IvRzgNZE3GLjVurBj"
    "evjlhpBxTvxlU+hNsqsvLlqAzwgzwoWbnBVmZBe/gpO4LLBEl6VhF3lyNGSseDfGn5HvZJEqB2N3DB0G1rD2flvLZTB2LXa9OEku"
    "6HuQQT8XYqjnvU0Qaedwkkl/kpm+GBmFETrfvv2ZOKY6Jy266I/77zfnb8Nl9+L89/M/zP6Oy9a4Bf6JU4g+Zs3exGY7pntXWDyM"
    "COWmCZZhM/DDYQk2//OPKTbHwckpuk3XrOyRTQrRi/x4zzzwj5ODPqj00xgyrh98L7SQ0+Cr2bsCV9/ZPUWpA+QF/M/JOYlL+d++"
    "9q/imPUuhFl2u9Ss273zeIq4THFzzlmx8fivGR4qTYR9MkyKmuKnLA2SCnOiVEnQ8BHIc8hWLMPnrUoGbysEakoh9DK7ja8gSdlh"
    "3DPD2TprggwH902BHfE+k5C//pG+MtvR17c/c044qDrtiL7sk573f74yKKMHe0wd4PhgX5NNF/x2jDQQr8G1/OnspjimYZ0IymMh"
    "jukkUKLpnWQRrKz9xu61RbHX1xYpvBS6xdmXXHVqqvVouuOggq1y0dRktmqNGjb5gT80z9xJvYGRk0yGW42gelYhhCxcgXVDQzZc"
    "hFXFqKlBL1XZfV6ZfEzqezTz9BRIpxthRMxELEwZKF6RRPS9E5U/Q7naX5p9I0sCT53OQG8FHv40rBFtYvg6u5woCfttX3E/NMtC"
    "HUjEYVrBkG0gwoCooGeLEjXo5yBjjh2fPH3Pu27qXR1gTP+cOVsHuP0q/rP83htOcLdSglZkpw+EoKUzYASp5DHNrMEubk2W4lwM"
    "tqW5JPlHIX0TZCULDEuu1LxkQDTw4M710sEPfB6tSyMfBOqZhu0STjh4AxEy5tVjNB24EyA6oW79tFsdI14xUNHv0nbge5+5n9pO"
    "NBpGXwS2+DZzHVyxzqyvEbZHLGMQcHNEYoZMNwCW8QfPp13IC9kSvC4L7DKvU8YnQJj4AxgzgZ5XL57nMfwzEPLSGB00yl+QLcZ9"
    "oZLpP/GhMuX2PjdbV8fUlsBy+CQtqpewR1CuHpUcB/l1Zh/UsJvw2sN2wi+x8F6kjDSvqVQssy/w/fhZAHQWeyrleQ8yG31B25E4"
    "9qXT9NnOP0Vlizkwip0fTa+F/fAohc9sC/fiwejRI+9HMH7KcgZm90QSczqXRSvVs1l9UrlBpXrprnP4B/s53XU+/jGqUq6AJu3X"
    "m7W60j5U/jwV3Wvec7MvjKf5GDJwSlZz9DObcFuUe4EfSV4zM3xOOZzEcVqnbHZVG+MkXUGm7odDKRkBoZLLDOHpvD3Mp4JYERoh"
    "NB7Kz/QSkjVQloaA7kEhR0PSl2mT3tT5gj6izF+ivqBTKbPJSE9k1Dh39jWSed3C+hzzIiDXidqOvsGe/oZeAB8SAuBqlzpF5IqH"
    "BEDQAG5QI0oGBG3oBqU3epeWCqU3Qqvf2W4AUq8tqyfRu7ZkoPQyvn7oxlfIkkzi78JNjWxOVVoKg6EkK63ysbNXKZY0Zclizimg"
    "KNNgLOcj7rDJV3JPzUi6TTalw8lkasnwtgsnUVXseEaeDBkWu0Py/D4ZjcT5F7c73h35keBnO/SZOzCkzJdFCaSIF26o9FmvUNIp"
    "JJzKd0Y9iF8S+zA/ZvwHzsKgKQ0VMc9NoTJS3Zh9MPsfkNlgqj9Qzrm8TAbjumC5jxkdbQigOp213/DK2M+ys3MMqDODqL/fheua"
    "GX2fLmO5DVt4FfgqTU7ieacTHEaD5ROlKpYKDJVK7GPO/JaFT9N5D+U8iSI+D3Bx2zracsLdsVy+iPmF6d9P3+RO13fC3fKxGZxL"
    "8ac1vqdeb6i/yCRGoTTeKUQYetCjhr0MaURsZJBApWqjfHTIl0hSDdS4pZKwpYqASg33NaTdm8RqN7AkodYvFdxlM0Zxqzwcwomk"
    "q56P18CSJCs/MESmmpybKp4GjP7e7EvmfiCkZO1dY6AS2jACpalu6hmTcG45zmFHJZJbgArJ/FfK1o3pkMotG52wRv1dtGyOLKFV"
    "56BOFdrZWbIijHQ4a3eSr+tU566kK7coJTR13g3gphoaq81Gxl5wnCfyq9n7yBpYQj/ThZejXy63jv11ETWvLRRBq54lcsCTnnXV"
    "RKYzabBKtZlbswXql+X9yUWZBAOVAQh5EybDQ66ErxhuDOcCJfOpUCZsWO8sLhy7edk8AmSmce1vwQFhG8hLFa+08YlT1lFBiqK+"
    "l+tngQQ1xlg2wyVu2dxEm4mMjOfvss+s2boyKh1CRvf23bax5XCyJcwyqzPNnKKzmis01uA5W71gCY0PChVepedV+bzOBfQJgS3z"
    "i4dRX+ByWRJRVC+8SzFl4pjikoxx9cVMZoSOCtmdsvHDURD19JXHxfXKF63BxLJHGv5Qr1lM6zcyKymrG7pKYRHfWfrSMAh0zxCt"
    "pPy+PPtUMi1lPH6LMh0bwKHU6KppRaU2/4PEMNiNNqOXKW+nQHDtG2UMl3P5i+RHNyLvE67MfP7jWZH5fCq574vWSmz579ppdnNy"
    "gFL6cg6wo0znH875WhVK/4byP8//7EjnBcWA7vQcYxVYrsRimpeK1R8lFie4a+vys7auoikpvk+6KNMeFgQ7OflDGq8456VHOTkp"
    "Y95bNp4XUY5TLNvE0+hrgok09RXFvlJpXY+yLsxsWPSGw1QrVuyxDChVVC6JIF5Ac3MghKVNcuqLQi7WoXdJEaXJbP/lOAZoXK/i"
    "uECnuMWu8RHnHjf9SBQTk/1ywp1RQjXjUnjxzniu3EntaXlzROcQZju5bKrz53MFaznLDGA838U0p0lbOfK33ri3vnovvePZo7Xx"
    "1/n7Vfx5lz7+YqV5bYjGWotTqXKWRiuoMZ/ryPTOWp3S7K3R7EkYC0hujlNxgYwg/B5zujtJip5jA2mjS55FO0FC0cO8YitDhcwN"
    "RSAZKsjEIKTKgwCURV5biyLAvk3NOqy86Ee1rZFCJBNR69zYlC/8Ou9BJm3oStiEjo8I2vMMQ7u04aTNXpI+tih3RHFYqlFhb5a9"
    "ZTY/THda/LnU2RLj18mQQ5Zp07+Ecb1M+dQFEmfRfohdmwx/uKukDQvvAPPBDaX067O5MhdA4Zg7t7B4oG5xuHexsrIEyCojF+03"
    "/4b/rtNS+qCKb+/qEiN18ndaR80cEL0kFzBPyTTexarKEbIQHZCnZSGeOP/iLogKpngHzPx7SuM/f1u7gOiQnPB8BxP+jkoan2aq"
    "179jIX/9QQgj3bwoSzgv9PVpM2s63UwryDmLpoVRSoqWb/O5IS38VQI5iOtpun26j7E8V35Zvvd0D1ygc9xZiFJyqPxMpD/nkf3I"
    "BXvvcAaPhf2mc2pCEXBPJqCS2frZ4KJj/D+QV0T+4gGrwOnTm3cl4BXOhY5ZdFlpYBYL28qC2PgiDHOXzwrKtnrmBMhNqlVItOPs"
    "UFx8Cl0GTzp0gwDYqOiAE8/IG0KZOBCZXBuD8a4gs0pB/rpZX0hRkTlfWViqTnIOffbg3iB5zE5XNrTlbzoQi6iwFJFuOsSan5RJ"
    "29IYobzKaqcAsr8uNI9Pp7HZoJmqOr1Qaf4hu0Y2n/yqTAqWRDvlhqcSPNnJpCoqHkoFNN39KI1QV9I5OMjSuB/Yt02iXCYrY+vm"
    "yOAF1AOUSmkmrHx7uo1i1TD4PE23BWaEq9jVVbYAXXFzRWQR9lu5vY67skL5895Ci5BpHi3bu2RiS8EFcqXQBJOt/RY5jt+FOGbZ"
    "KeBokj/SQacS9PM+/Hs7eagUAZyDzNEdKhg7R5/fruTOib+hX8DRfi3dHfm22j/VDX9qLOmF8SDXpp7h6tqnmCyZf83I3JPNik3K"
    "zH6msG8KSwyzeyLEWW0jSLzDeW4An27h/NOWKk8js62rrNpnAjnMAKRTFpKDfnHma6L2DeUb/wxeEbX5vMRbGR/OvzJ/QQ6PW2Pw"
    "ikb/Rcj9ShxzASKdT+jwnifxHX2OQVvYhwZHlYklIxwxt1pzkYmundSrIsxsXd9TNtnpwpjRzXxhpUnFirWyb0v7UcPh9lmCsZIM"
    "ZPka+tlbpp1JV8sa39t0kJifTcqye6XaDZDdLiX9FQ6HzumoVPQEyMwQnK6nrfQdWJabaaqN9PLFXaq3JUGqeV6RUaquGaza3Tni"
    "nXYRa0jN3o7Iz8lKEh52sk/jF0KUR6FjJi+8k7f4rGH3TXidu8YVK1pmfcpnybkVKKTJzHdHPxpRapMicxnFrxDADC07h0Ymosso"
    "5KaJIwsCm1CiIzQdLNcf8OQZqqEhPZUxSQfFSoH8aPZ3Mi4/h1x/n5QuZTI65QxmZsmrWDd69LhsbQcqwRuA/GUp0xQil0u9c2qr"
    "Mr8s4hdJP6tZJle8U7KVJisiFWi4NXXHhulr3adJD8/eJcWHvdaldrg66b9JR67kr0aklQXH2JHkAO4+neCdeII2vkpuI9V+hWNd"
    "+ZS7JpSp7kJVCjaufWtN6RBI/SW3wOwmVqd0pXCCN9pRLRK18uncehwUbN75LRhtBb7tddrPo3vFt9m3nldou1d0eSkICsAip5aB"
    "4l7qpGILolVqVdr+hhQz2csCN5Vy+qQ/cpg5dUoQ4WjRPR3kuAQTGbBM+cuk+QXB8LUMBFiOAx+70C+xcSAffE4cIPgyH5crvv+I"
    "dBX5RZ3c4ys8IqXo5m/uLw1edFjYfHkaJDPPF6pxKw6SoaHm8CjHoYSd9HSlLGSLt8ItjMJZdkmiz7KX3217lHZ9aSd0fzC/mxju"
    "2EVtCmKGTOzyk2gzqCCttfwIGk5KUyxBqn7O0XZP4V/14Yf1TyQawpZ+q/M1IyJBjpi6y+339IpFpirmZzO5qVVTCfwUFH8rrT0K"
    "YaWxLkF2DA2FIdP9aXqRduWV6fqqMVNLj5W1PERw0dqWHltYVDeZYdlj+wW4+R9cCEvg1vxNU8eX6dMUVLrXqCiKCl+x9nZDk7KU"
    "ZFEgNxWZGjYdS5b2lV7pvXDZeS8By91F0L80RCBKi4zz4L+x/wfRNfmYgjLLZPGtptxay9LKpk/fZ1VlGnUGOFtUMm9t6dKyqst6"
    "pYpHxjq8hdOrjL3SBZaqFNio7Vale7kNZXm47XedRSOcbPYeB5PRT3SebiqNS2iFcEO+ftYdUP7U7PuiKdydTPpL5tAE4VJdLQw5"
    "4edBLNcnbziZZibPf1CXqdemR+v6utrsySKQZZlN0DCSwuBCwg7WFAwuckpVOBv/ZL/d2uiY1wHgaIs57eCxfXOcvC6pzyX1t8DZ"
    "qX5yoc+pPRB5g8wabpv8Zmx3tP1Op3ALnYI/OazjVbFpQehr55O05ChNdHr1f+THsEW0MN2isWx6eci8+7N+wmGn5cA16bZsMuY7"
    "AKXqhQ7gCPmGDBsV+FreAdN/K5uIAmiagep6rQje/Bnp9AQeHbLl9EF5bz01lJ6dzMD4LQO7c7IDEScSInShMLrcusE3GugbWoeS"
    "g/+/f1Qxm5LWC2ZQykOewjQ7dh4V5Nlu2FQ4sXY4TraYy8Esr37SBX9O3Kg5tvh7OaU/QVFYhATIWolFzVBiohN5UP85ra3sZmex"
    "W7r/t2g3pltv+FV0pUy+nF4xfVke1rQ4o4qDPVclM5ZtSkNPctdmWZ79fwcHfgo="
)
W = types.ModuleType("week01_support")
exec(zlib.decompress(base64.b64decode(_SRC)).decode("utf-8"), W.__dict__)

raw = W.load_raw()
print(raw.shape)
raw.head(10)

<div dir="rtl" align="right">

## الجزء أ: NumPy بلا حلقات

القاعدة: لا حلقة `for` على العناصر أو النقاط. استخدم البث (broadcasting) والفهرسة والدوال المتجهة. في A6 فقط تُسمح حلقة على عدد المراكز إن احتجت، والأفضل بدونها.

</div>

<div dir="rtl" align="right">

### A1. توحيد الأعمدة (z-score)

</div>

In [ ]:
def zscore(X):
    """Standardize each column: (X - mean) / std, computed over axis 0.
    A column whose std is 0 becomes all zeros (no nan, no inf)."""
    # Your code here
    raise NotImplementedError

W.check_zscore(zscore)

<div dir="rtl" align="right">

### A2. مصفوفة المسافات المربعة

الفكرة: ‖a − b‖² = ‖a‖² + ‖b‖² − 2 a·b. احسب كل الأزواج دفعة واحدة، ثم قصّ القيم السالبة الصغيرة الناتجة عن التقريب إلى صفر.

</div>

In [ ]:
def pairwise_sq_dists(A, B):
    """A: (n, d), B: (m, d) -> (n, m) squared Euclidean distances, never negative."""
    # Your code here
    raise NotImplementedError

W.check_pairwise(pairwise_sq_dists)

<div dir="rtl" align="right">

### A3. softmax مستقرة عددياً

</div>

In [ ]:
def softmax(Z, axis=-1):
    """Numerically stable softmax along `axis`. Must not overflow for inputs like 1000."""
    # Your code here
    raise NotImplementedError

W.check_softmax(softmax)

<div dir="rtl" align="right">

### A4. الترميز الأحادي (one-hot)

</div>

In [ ]:
def one_hot(y, num_classes):
    """y: (n,) integer labels -> (n, num_classes) matrix of 0/1."""
    # Your code here
    raise NotImplementedError

W.check_one_hot(one_hot)

<div dir="rtl" align="right">

### A5. المتوسط المتحرك بحيلة المجموع التراكمي

الناتج بطول n − k + 1، مثل `np.convolve(..., mode="valid")`، لكن بـ`np.cumsum` وطرح واحد.

</div>

In [ ]:
def moving_average(x, k):
    """Mean of every window of length k (valid windows only), using np.cumsum."""
    # Your code here
    raise NotImplementedError

W.check_moving_average(moving_average)

<div dir="rtl" align="right">

### A6. خطوة واحدة من k-means

عيّن كل نقطة لأقرب مركز (استخدم A2)، ثم احسب كل مركز جديد كمتوسط نقاطه. المركز الذي لا نقاط له يبقى كما هو.

</div>

In [ ]:
def kmeans_step(X, C):
    """X: (n, d) points, C: (k, d) centroids -> (labels (n,), new_C (k, d))."""
    # Your code here
    raise NotImplementedError

W.check_kmeans_step(kmeans_step)

<div dir="rtl" align="right">

## الجزء ب: تنظيف طلبات متجر كتب عربي

البيانات في `raw`: نحو ١٬٢٦٠ صفاً، كل الأعمدة نصوص كما تصل من نظام حقيقي.

| العمود | ماذا ستجد فيه |
| --- | --- |
| `order_id` | مسافات زائدة، حروف صغيرة، وعلامة اتجاه مخفية (U+200F) |
| `order_date` | ISO، ويوم/شهر/سنة، وأرقام هندية، ووقت ملحق، وتاريخ هجري ينتهي بـ"هـ" |
| `city` | الرياض، رياض، Riyadh، RUH، الريـــاض... وقيم لا تُسترجع مثل "؟؟" |
| `book_title` | تشكيل، وتطويل، وهمزات مختلفة، ومسافات، وعلامات مخفية |
| `price` | "٤٥٫٥ ر.س"، "SAR 45.50"، "45.50 ريال"، "مجاني"، أو فارغ |
| `quantity`، `rating` | أرقام هندية، و"4.0"، وقيم خارج النطاق، ونصوص |
| `updated_at` | وقت آخر تعديل؛ يحدد أي نسخة من الطلب المكرر هي الأحدث |

ابدأ بفحص البيانات بعينك قبل أي كود تنظيف.

</div>

In [ ]:
raw.info()
raw.sample(8, random_state=1)

In [ ]:
raw["city"].value_counts().head(40)

<div dir="rtl" align="right">

### B1. الأرقام الهندية إلى أرقام لاتينية

حوّل ٠–٩ (العربية) و۰–۹ (الفارسية) إلى 0–9، والفاصلة العشرية العربية `٫` إلى نقطة، واحذف فاصل الآلاف `٬`. القيم المفقودة تبقى مفقودة.

تلميح: `str.maketrans` مع `Series.str.translate`.

</div>

In [ ]:
def to_ascii_digits(s):
    """Arabic-Indic and Persian digits -> 0-9, '٫' -> '.', remove '٬'. NaN stays NaN."""
    # Your code here
    raise NotImplementedError

W.check_to_ascii_digits(to_ascii_digits)

<div dir="rtl" align="right">

### B2. تنظيف السعر

الناتج float بالريال. "مجاني" = 0. الفارغ وغير المفهوم = NaN (ستعوّضه في B8).

</div>

In [ ]:
def clean_price(s):
    """Text prices -> float SAR; missing or unreadable -> NaN; 0.0 when the text says مجاني"""
    # Your code here
    raise NotImplementedError

W.check_clean_price(clean_price)

<div dir="rtl" align="right">

### B3. التواريخ الميلادية بأربع صيغ

الصيغ الأربع:
- <code dir="ltr">2026-03-01</code>
- <code dir="ltr">01/03/2026</code> (يوم/شهر/سنة)
- <code>٢٠٢٦-٠٣-٠١</code> (أرقام هندية؛ تُقرأ من اليمين: السنة ثم الشهر ثم اليوم)
- <code dir="ltr">2026-03-01 14:22</code> (احذف الوقت)

التاريخ الهجري يبقى `NaT` هنا.

تلميح: لا تعتمد على تخمين pandas للصيغة؛ حدّد `format` لكل صيغة، وتأكد أن <code dir="ltr">01/03/2026</code> تعني ١ مارس.

</div>

In [ ]:
def parse_dates(s):
    """Gregorian dates in 4 formats -> datetime64 (date only). Hijri strings -> NaT."""
    # Your code here
    raise NotImplementedError

W.check_parse_dates(parse_dates)

<div dir="rtl" align="right">

### B3+. تحدٍّ إضافي: التاريخ الهجري

حوّل القيم التي على شكل <code dir="ltr">1447-09-13هـ</code> (أو بأرقام هندية) إلى ميلادي بتقويم أم القرى، باستخدام `from hijridate import Hijri`. باقي القيم `NaT`.

</div>

In [ ]:
def hijri_to_gregorian(s):
    """'YYYY-MM-DDهـ' (Umm al-Qura) -> Gregorian datetime64. Other values -> NaT."""
    # Your code here (pip install hijridate)
    raise NotImplementedError

W.check_hijri(hijri_to_gregorian)

<div dir="rtl" align="right">

### B4. توحيد النص العربي

بالترتيب: احذف العلامات المخفية (U+200B–U+200F، وU+202A–U+202E، وU+2066–U+2069، وU+FEFF)، ثم التشكيل (U+0610–U+061A، وU+064B–U+065F، وU+0670، وU+06D6–U+06ED)، ثم التطويل (U+0640). وحّد أ وإ وآ وٱ إلى ا، واجمع المسافات المتتالية في مسافة واحدة.

**لا تغيّر** ة إلى ه ولا ى إلى ي: هذا يغيّر الكلمات نفسها. اكتب في سجلك لماذا.

</div>

In [ ]:
def normalize_arabic(s):
    """Remove invisible marks, diacritics and tatweel; unify alef forms; collapse spaces.
    Keep ة/ه and ى/ي unchanged. NaN stays NaN."""
    # Your code here
    raise NotImplementedError

W.check_normalize_arabic(normalize_arabic)

<div dir="rtl" align="right">

### B5. أسماء المدن الموحدة

الأسماء المعتمدة ستة: الرياض، جدة، الدمام، مكة المكرمة، المدينة المنورة، أبها. كل ما عداها، مثل "؟؟" و"N/A"، يصبح NaN.

تلميح: طبّق `normalize_arabic` ثم `str.lower`، وابنِ قاموساً من الأشكال الموحدة إلى الاسم المعتمد. خلية `value_counts` في الأعلى تريك كل الأشكال.

</div>

In [ ]:
def canonical_city(s):
    """Any spelling of the 6 cities -> its official Arabic name; anything else -> NaN."""
    # Your code here
    raise NotImplementedError

W.check_canonical_city(canonical_city)

<div dir="rtl" align="right">

### B6. رقم الطلب والتكرار

1. `clean_order_id`: احذف العلامات المخفية والمسافات، وحوّل إلى حروف كبيرة.
2. `dedupe_orders`: يستقبل جدولاً فيه `order_id` منظف و`updated_at` من نوع datetime، ويُبقي لكل طلب أحدث نسخة فقط.

سؤال للسجل: ماذا يحدث لو حذفت التكرار **قبل** تنظيف رقم الطلب؟

</div>

In [ ]:
def clean_order_id(s):
    """Remove invisible marks and spaces, upper-case: ' ord-100123\u200f' -> 'ORD-100123'."""
    # Your code here
    raise NotImplementedError

def dedupe_orders(df):
    """Keep only the latest row (by updated_at) for each order_id."""
    # Your code here
    raise NotImplementedError

W.check_dedupe(clean_order_id, dedupe_orders)

<div dir="rtl" align="right">

### B7. الكمية والتقييم

الناتج من نوع `Int64` (يقبل القيم المفقودة). الكمية الصالحة عدد صحيح من ١ إلى ٥٠، والتقييم من ١ إلى ٥. كل ما عدا ذلك مفقود، مثل: <code dir="ltr">"0"</code> و<code dir="ltr">"-1"</code> و<code dir="ltr">"120"</code> و<code dir="ltr">"2.5"</code> و"اثنان" و"ممتاز".

</div>

In [ ]:
def clean_quantity(s):
    """Whole numbers 1..50 -> Int64; anything else -> <NA>."""
    # Your code here
    raise NotImplementedError

def clean_rating(s):
    """Whole numbers 1..5 -> Int64; anything else -> <NA>."""
    # Your code here
    raise NotImplementedError

W.check_quantity_rating(clean_quantity, clean_rating)

<div dir="rtl" align="right">

### B8. تعويض السعر المفقود

عوّض السعر المفقود بوسيط سعر الكتاب نفسه (`book_id`). لا تحذف أي صف.

ملاحظة مهمة للأسبوع ٤: هنا نحسب الوسيط من كل البيانات لأننا ننظف فقط. عند تدريب نموذج، يُحسب من بيانات التدريب وحدها داخل Pipeline، وإلا تسرّبت معلومات الاختبار.

</div>

In [ ]:
def impute_price_by_book(df):
    """Fill missing price with the median price of the same book_id. Same number of rows."""
    # Your code here
    raise NotImplementedError

W.check_impute(impute_price_by_book)

<div dir="rtl" align="right">

### B9. خط التنظيف الكامل

اجمع كل ما سبق في دالة واحدة، بهذا الترتيب:
1. نظّف `order_id` وحوّل `updated_at` إلى datetime، ثم احذف التكرار.
2. ابنِ الأعمدة: `order_date` (مع الهجري إن أنجزت B3+)، و`city`، و`book_id`، و`book_title_norm`، و`price`، و`quantity`، و`rating`.
3. عوّض السعر، ثم `revenue = price × quantity` (مفقود إن كانت الكمية مفقودة).
4. رتّب حسب `order_id`، وأعد ترقيم الفهرس، والأعمدة بترتيب `W.FINAL_COLUMNS`.

</div>

In [ ]:
def clean_orders(raw):
    """raw orders -> one clean row per order, columns in W.FINAL_COLUMNS order."""
    # Your code here
    raise NotImplementedError

W.check_clean_orders(clean_orders)

<div dir="rtl" align="right">

### B10. أسئلة تحليلية

1. `monthly_city_revenue`: جدول صفوفه الأشهر بصيغة نص `YYYY-MM`، وأعمدته المدن، وقيمه مجموع الإيراد (والشهر بلا مبيعات = 0). تجاهل الصفوف بلا تاريخ أو بلا مدينة.
2. `top_books`: أعلى `n` كتب إيراداً، كسلسلة مفهرسة بـ`book_title_norm` ومرتبة تنازلياً.

الفحص يشغّل دوالك على الجدول الصحيح، حتى لا يتأثر بأخطاء سابقة.

</div>

In [ ]:
def monthly_city_revenue(df):
    """Rows: 'YYYY-MM' strings, columns: cities, values: summed revenue (0 when no sales)."""
    # Your code here
    raise NotImplementedError

def top_books(df, n=5):
    """Top-n books by total revenue: Series indexed by book_title_norm, descending."""
    # Your code here
    raise NotImplementedError

W.check_monthly_city_revenue(monthly_city_revenue)
W.check_top_books(top_books)

<div dir="rtl" align="right">

### B11. تحقق من المخطط بنفسك

اكتب دالة ترفع `AssertionError` برسالة واضحة إذا خالف الجدول أياً من هذه القواعد:
- `order_id` موجود دائماً وغير مكرر.
- `price` موجود دائماً وغير سالب.
- `quantity` من ١ إلى ٥٠ أو مفقودة، و`rating` من ١ إلى ٥ أو مفقود.
- `city` أحد الأسماء الستة أو مفقودة.
- `revenue` يساوي `price × quantity`، ومفقود حين تكون الكمية مفقودة.

الفحص يمرّر لها جدولاً سليماً (يجب ألا تعترض) وستة جداول معطوبة (يجب أن تكتشفها كلها). في الأسبوع ٥ ستعيد كتابتها بمكتبة pandera.

</div>

In [ ]:
def check_schema(df):
    """Raise AssertionError with a clear message when a rule is broken."""
    # Your code here
    raise NotImplementedError

W.check_schema_fn(check_schema)

<div dir="rtl" align="right">

### B12. ارسم ما وجدته

لا فحص هنا. ارسم الإيراد الشهري لكل مدينة، وأعلى خمسة كتب إيراداً. الدالة `W.ar()` تجعل matplotlib يرسم العربية بالاتجاه والأشكال الصحيحة (بعد تثبيت الأداتين في أول الورقة). اكتب تحت كل رسم خلاصته في جملة.

</div>

In [ ]:
import matplotlib.pyplot as plt

try:
    clean = clean_orders(raw)
except NotImplementedError:
    clean = None
    print("أكمل B9 أولاً")

if clean is not None:
    # Your plots here: monthly revenue per city, then top-5 books
    pass

<div dir="rtl" align="right">

## الجزء ج: أسئلة للسجل

أجب في خلية Markdown تحت كل سؤال، بجملتين أو ثلاث:

1. أي خطوة في التنظيف تُفقد معلومات بلا رجعة؟ وكيف توثّقها؟
2. احسب: كم طلباً كان سيُحسب مرتين لو حذفت التكرار قبل تنظيف `order_id`؟
3. لو أردت تدريب نموذج يتنبأ بالتقييم من هذه البيانات، اذكر مصدرين محتملين لتسريب البيانات.
4. ما البطاقات الخمس التي ستضيفها إلى Anki من هذه الورقة؟

</div>

In [ ]:
W.progress()